# Lesson 2.3 - Errors, Formatting, and JSON

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Catch and handle script errors deliberately
- Format data for a human or a machine consumer
- Write JSON output that another automation step can read

**Project file(s) produced:** `scripts/lesson-2.3-errors-formatting-json.ps1` and `artifacts/report.json`

## Concepts

Good automation does not fail silently. `try`/`catch` lets you trap failures, and `finally` lets you clean up or log status whether the command succeeded or failed.

PowerShell can also output for two audiences: humans and machines. `Format-Table` is useful for a person reading the console, while JSON is a better handoff format for another script or service.

Reference: [PowerShell Documentation](https://learn.microsoft.com/en-us/powershell/)

In [ ]:
from pathlib import Path
import shutil
import subprocess
import textwrap

lesson_dir = Path('study-plan/topic-powershell/phase-2-scripting-mechanics')
artifact_dir = lesson_dir / 'artifacts'
artifact_dir.mkdir(parents=True, exist_ok=True)

def run_powershell(script_text: str, script_name: str) -> int:
    script_path = lesson_dir / script_name
    script_path.write_text(script_text, encoding='utf-8')
    executable = shutil.which('pwsh') or shutil.which('powershell')
    if executable:
        result = subprocess.run([executable, '-NoLogo', '-NoProfile', '-ExecutionPolicy', 'Bypass', '-File', str(script_path)], capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
        return result.returncode
    print(f'{script_path.name} written. Run it in PowerShell to see the live output.')
    return 0

## Code-Along: Step 1 - Trap a predictable failure

This step intentionally asks for a file that does not exist. The script uses `try`/`catch` so you can see how a controlled failure is different from a crash.

In [ ]:
script = textwrap.dedent(r'''
$ErrorActionPreference = 'Stop'
try {
    Get-Content -Path (Join-Path $PSScriptRoot 'artifacts/does-not-exist.txt')
} catch {
    Write-Host "Caught expected error: $($_.Exception.Message)"
}
''')
run_powershell(script, 'lesson-2.3-try-catch.ps1')

## Code-Along: Step 2 - Emit machine-readable output

Structured output is easier for a pipeline to consume than console text. Here the script writes a JSON report with the fields a later step would want to inspect.

In [ ]:
script = textwrap.dedent(r'''
$report = [pscustomobject]@{
    Name      = 'PowerShell Study Plan'
    Status    = 'Healthy'
    Timestamp = (Get-Date).ToString('o')
}
$jsonPath = Join-Path $PSScriptRoot 'artifacts/report.json'
$report | ConvertTo-Json -Depth 3 | Set-Content -Path $jsonPath
Get-Content $jsonPath
''')
run_powershell(script, 'lesson-2.3-json-report.ps1')

## Exercise

Add a second object to the JSON report so the file contains a `Summary` property and a `Findings` array.

**Your task:**
1. Expand the JSON object.
2. Save the updated structure to the same file.
3. Confirm the JSON still parses cleanly.

**Expected result:** a report that is easy for another automation step to read.

In [ ]:
# TODO: add Summary and Findings to the JSON report
exercise_script = textwrap.dedent(r'''
Write-Host "TODO: enrich the report object"
''')
run_powershell(exercise_script, 'lesson-2.3-exercise.ps1')

## Key Takeaways

- `try`/`catch` makes failures visible and intentional.
- Formatting is for humans; JSON is for machines.
- A small report object is often enough to bridge one automation step to the next.

---

**Next:** [Lesson 3.1 - Modules and the Gallery](../phase-3-modules-automation/lesson-3.1-modules-gallery.ipynb)